# Human time-travel round-trip consistency (A→B→A) — cache + figures

This notebook adds a **high-impact marketing metric** that is hard to express with single-direction outcome profiles:

> If you time-travel from release **A** to **B**, can you come back to **A** and recover the original identifier?

This is a **round-trip consistency** test for the Ensembl gene backbone.

## Why this markets IDTrack

- It makes the time axis tangible as a *reproducibility coordinate system* rather than a “latest lookup”.
- It is naturally square (A×B) and can be reported as a heatmap.
- It exposes irreversibility and split/merge behavior without claiming biological “accuracy”.

## Dependencies

- Requires the pools cache from `00_build_time_travel_matrix_cache.ipynb` (for reproducible ID sampling).
- Uses `IDTRACK_LOCAL_REPO` caches under `idtrack/docs/_notebooks/idtrack_cache/`.

## Outputs

- `_outputs/_publication/figures/fig_time_travel_roundtrip_matrix_human.pdf`
- `_outputs/_publication/figures/fig_time_travel_roundtrip_delta_curves.pdf`
- `_outputs/_publication/figures/fig_time_travel_roundtrip_directional_decay.pdf`
- `_outputs/_publication/tables/time_travel_roundtrip_summary.csv`
- `_outputs/_publication/tables/time_travel_roundtrip_worst_pairs.csv`

Figures are also mirrored under:

- `idtrack/reproducibility/experiments/_outputs/time_travel_matrix/`


In [ ]:
from __future__ import annotations

import json
import time
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

try:
    import seaborn as sns
except Exception:  # noqa: S110
    sns = None

import os
import sys

# Add experiments/src to sys.path (layout-aware; works on Slurm and locally)
REPO_ROOT = Path(os.environ.get('REPO_ROOT', Path.cwd())).expanduser().resolve()
while REPO_ROOT != REPO_ROOT.parent and not (
    (REPO_ROOT / 'idtrack').is_dir()
    and ((REPO_ROOT / 'reproducibility').is_dir() or (REPO_ROOT / 'idtrack' / 'reproducibility').is_dir())
):
    REPO_ROOT = REPO_ROOT.parent

REPRO_ROOT = REPO_ROOT / 'reproducibility' if (REPO_ROOT / 'reproducibility').is_dir() else REPO_ROOT / 'idtrack' / 'reproducibility'
EXPERIMENTS_SRC = REPRO_ROOT / 'experiments' / 'src'
if str(EXPERIMENTS_SRC) not in sys.path:
    sys.path.insert(0, str(EXPERIMENTS_SRC))

from experiments_utils import (  # noqa: E402
    MANUSCRIPT_COLORS,
    atomic_write_dataframe_csv,
    notebook_context,
    read_json,
    read_pickle,
    save_figure,
    stable_hash,
    write_pickle,
)

ctx = notebook_context("time_travel_matrix", start=REPO_ROOT)
CACHE_DIR = ctx.experiment_cache
MANUSCRIPT_FIGURES = ctx.manuscript_figures
MANUSCRIPT_TABLES = ctx.manuscript_tables

print("CACHE_DIR:", CACHE_DIR)
print("IDTRACK_LOCAL_REPO:", ctx.idtrack_local_repo)


In [ ]:
# -------------------- Locate pools cache (produced by 00_build_time_travel_matrix_cache) --------------------

candidates = list(CACHE_DIR.glob("time_travel_matrix_grid_*.pickle"))
if not candidates:
    raise FileNotFoundError(
        f"No time-travel grid cache found under {CACHE_DIR}. Run: 00_build_time_travel_matrix_cache.ipynb"
    )

RESULTS_PKL = max(candidates, key=lambda p: p.stat().st_mtime)
fp = RESULTS_PKL.stem.split("_")[-1]

PARAMS_JSON = CACHE_DIR / f"time_travel_matrix_params_{fp}.json"
POOLS_JSON = CACHE_DIR / f"time_travel_matrix_pools_{fp}.json"

if not POOLS_JSON.exists():
    raise FileNotFoundError(
        f"Missing pools cache {POOLS_JSON}. Re-run: 00_build_time_travel_matrix_cache.ipynb"
    )

params = read_json(PARAMS_JSON) if PARAMS_JSON.exists() else {}
pools_payload = read_json(POOLS_JSON)
pools = pools_payload.get("pools", {})

display(pd.DataFrame([params]) if params else pd.DataFrame())
print("Using pools:", POOLS_JSON)


In [ ]:
# -------------------- Configuration (manageable; cache-first) --------------------

ORGANISM_ALIAS = str(params.get("organism_alias", "human"))
SNAPSHOT_RELEASE = int(params.get("snapshot_release", 114))

# Use the same release grid as the time-travel matrix cache by default.
FROM_RELEASES = [int(x) for x in (params.get("from_releases") or [])] or list(range(75, 115, 5))
TO_RELEASES = [int(x) for x in (params.get("to_releases") or [])] or list(range(75, 115, 5))

# Sampling / compute budget
N_IDS_PER_PAIR = 200
N_BOOTSTRAPS = 3
RANDOM_SEED = 0

# Round-trip is defined on the Ensembl gene backbone
FINAL_DATABASE = None
STRATEGY = "all"

PARAMS_RT = {
    "organism_alias": ORGANISM_ALIAS,
    "snapshot_release": SNAPSHOT_RELEASE,
    "from_releases": FROM_RELEASES,
    "to_releases": TO_RELEASES,
    "n_ids_per_pair": N_IDS_PER_PAIR,
    "n_bootstraps": N_BOOTSTRAPS,
    "random_seed": RANDOM_SEED,
    "strategy": STRATEGY,
    "final_database": "Ensembl gene",
    "pools_fp": fp,
}

FP_RT = stable_hash(json.dumps(PARAMS_RT, sort_keys=True), n=12)
RT_PKL = CACHE_DIR / f"time_travel_roundtrip_{FP_RT}.pickle"

print("RT_PKL:", RT_PKL)
display(pd.DataFrame([PARAMS_RT]))


In [ ]:
# -------------------- Compute round-trip cache (or load) --------------------

if RT_PKL.exists():
    rt = read_pickle(RT_PKL)
    print("Loaded:", RT_PKL)
else:
    import idtrack

    rng = np.random.default_rng(RANDOM_SEED)

    api = idtrack.API(local_repository=str(ctx.idtrack_local_repo))
    api.configure_logger()

    organism, latest = api.resolve_organism(ORGANISM_ALIAS)
    snapshot = int(SNAPSHOT_RELEASE)
    if snapshot > int(latest):
        raise ValueError(f"snapshot_release={snapshot} exceeds latest={latest} for {ORGANISM_ALIAS}")

    if max(FROM_RELEASES + TO_RELEASES) > snapshot:
        raise ValueError(f"Release grid exceeds snapshot={snapshot}: max={max(FROM_RELEASES + TO_RELEASES)}")

    print(f"Building/loading graph: {organism} snapshot_release={snapshot}")
    api.build_graph(organism_name=organism, snapshot_release=snapshot, calculate_caches=True)

    def _sample_ids(from_release: int, *, seed: int) -> list[str]:
        pool = pools.get(str(int(from_release))) or pools.get(int(from_release)) or []
        pool = [str(x) for x in pool]
        if not pool:
            return []
        r = np.random.default_rng(seed)
        k = min(int(N_IDS_PER_PAIR), len(pool))
        return r.choice(pool, size=k, replace=False).tolist()

    rows = []
    t0 = time.perf_counter()

    for b in range(int(N_BOOTSTRAPS)):
        seed_b = int(RANDOM_SEED) + 10000 + b
        for fr in FROM_RELEASES:
            ids = _sample_ids(int(fr), seed=seed_b + int(fr))
            if not ids:
                continue

            for to in TO_RELEASES:
                to = int(to)

                # Forward
                forward = api.convert_identifier_multiple(
                    ids.copy(),
                    from_release=int(fr),
                    to_release=to,
                    final_database=FINAL_DATABASE,
                    strategy=STRATEGY,
                    verbose=False,
                )

                # Round-trip per query: recover original ID in any back-mapped targets
                recovered = 0
                forward_success = 0
                forward_1_to_0 = 0

                for item in forward:
                    q = str(item.get("query_id"))
                    no_corresponding = bool(item.get("no_corresponding"))
                    no_conversion = bool(item.get("no_conversion"))
                    if no_corresponding or no_conversion:
                        forward_1_to_0 += 1
                        continue

                    targets = item.get("target_id") or []
                    targets = [str(t) for t in targets]
                    if not targets:
                        forward_1_to_0 += 1
                        continue

                    forward_success += 1

                    back_targets: set[str] = set()
                    for t in targets:
                        back = api.convert_identifier(
                            t,
                            from_release=to,
                            to_release=int(fr),
                            final_database=FINAL_DATABASE,
                            strategy=STRATEGY,
                            verbose=False,
                        )
                        bt = back.get("target_id") or []
                        back_targets |= {str(x) for x in bt}

                    if q in back_targets:
                        recovered += 1

                total = len(ids)
                rows.append(
                    {
                        "bootstrap": int(b),
                        "from_release": int(fr),
                        "to_release": int(to),
                        "n": int(total),
                        "forward_success": int(forward_success),
                        "forward_1_to_0": int(forward_1_to_0),
                        "roundtrip_recovered": int(recovered),
                    }
                )

    dt = time.perf_counter() - t0
    print(f"Computed round-trip cache in {dt/60:.1f} min")

    rt = {"params": PARAMS_RT, "rows": rows}
    write_pickle(rt, RT_PKL)
    print("Wrote:", RT_PKL)

rt_df = pd.DataFrame(rt.get("rows", []))
rt_df["frac_roundtrip"] = rt_df["roundtrip_recovered"] / rt_df["n"].replace(0, np.nan)
rt_df["frac_roundtrip_given_success"] = rt_df["roundtrip_recovered"] / rt_df["forward_success"].replace(0, np.nan)
display(rt_df.head())


In [ ]:
# -------------------- Aggregate over bootstraps + export table --------------------

if rt_df.empty:
    raise RuntimeError("Round-trip results are empty.")

# Aggregate counts across bootstraps, then compute fractions.
agg_rt = (
    rt_df.groupby(["from_release", "to_release"], as_index=False)
    .agg(
        n=("n", "sum"),
        forward_success=("forward_success", "sum"),
        forward_1_to_0=("forward_1_to_0", "sum"),
        roundtrip_recovered=("roundtrip_recovered", "sum"),
    )
)
agg_rt["frac_forward_1_to_0"] = agg_rt["forward_1_to_0"] / agg_rt["n"].replace(0, np.nan)
agg_rt["frac_roundtrip"] = agg_rt["roundtrip_recovered"] / agg_rt["n"].replace(0, np.nan)
agg_rt["frac_roundtrip_given_success"] = agg_rt["roundtrip_recovered"] / agg_rt["forward_success"].replace(0, np.nan)

out_csv = MANUSCRIPT_TABLES / "time_travel_roundtrip_summary.csv"
atomic_write_dataframe_csv(agg_rt, out_csv, index=False)
atomic_write_dataframe_csv(agg_rt, (ctx.experiment_outputs / "tables" / out_csv.name), index=False)
print("Wrote:", out_csv)

display(agg_rt.head())


In [ ]:
# -------------------- Figure: round-trip matrix heatmap --------------------

mat = agg_rt.pivot(index="from_release", columns="to_release", values="frac_roundtrip_given_success")
mat = mat.sort_index().sort_index(axis=1)

fig, ax = plt.subplots(1, 1, figsize=(7.5, 6.5), constrained_layout=True)
if sns is not None:
    sns.heatmap(
        mat,
        ax=ax,
        vmin=0,
        vmax=1,
        cmap="viridis",
        square=True,
        linewidths=0.25,
        linecolor=MANUSCRIPT_COLORS["grid"],
        cbar_kws={"label": "Round-trip recovery | forward success"},
    )
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right")
    ax.set_yticklabels(ax.get_yticklabels(), rotation=0)
else:
    im = ax.imshow(mat.values, vmin=0, vmax=1, cmap="viridis")
    fig.colorbar(im, ax=ax, label="Round-trip recovery | forward success")
    ax.set_xticks(np.arange(len(mat.columns)))
    ax.set_yticks(np.arange(len(mat.index)))
    ax.set_xticklabels([str(c) for c in mat.columns], rotation=45, ha="right")
    ax.set_yticklabels([str(i) for i in mat.index], rotation=0)

ax.set_title("Round-trip consistency on Ensembl backbone (A→B→A)")
ax.set_xlabel("to_release (B)")
ax.set_ylabel("from_release (A)")

written = save_figure(fig, "fig_time_travel_roundtrip_matrix_human.pdf", ctx, formats=("pdf",))
print("Saved:", written["pdf"])


In [ ]:
# -------------------- Figure: round-trip recovery vs |Δ release| --------------------

agg_rt2 = agg_rt.copy()
agg_rt2["abs_delta"] = (agg_rt2["to_release"].astype(int) - agg_rt2["from_release"].astype(int)).abs()

curve = (
    agg_rt2.groupby("abs_delta", as_index=False)
    .agg(mean=("frac_roundtrip_given_success", "mean"), std=("frac_roundtrip_given_success", "std"), n=("abs_delta", "count"))
)
curve["sem"] = curve["std"] / np.sqrt(curve["n"].replace(0, np.nan))

fig2, ax2 = plt.subplots(1, 1, figsize=(7.8, 4.2), constrained_layout=True)
ax2.errorbar(curve["abs_delta"], curve["mean"], yerr=curve["sem"], fmt="-o", ms=3, lw=1.4)
ax2.set_ylim(0, 1)
ax2.set_xlabel("|Δ release|")
ax2.set_ylabel("Round-trip recovery | forward success")
ax2.set_title("Round-trip consistency decays with release distance")

written2 = save_figure(fig2, "fig_time_travel_roundtrip_delta_curves.pdf", ctx, formats=("pdf",))
print("Saved:", written2["pdf"])


## Marketing extension: directional decay + worst-case pairs

Round-trip recovery is a **reproducibility diagnostic**: it quantifies how much information is lost when traversing release boundaries.

Two additions below make this easier to report:

- **Directional decay:** does going into the future vs the past behave differently?
- **Worst-case pairs:** concrete release pairs where irreversibility is strongest.


In [ ]:
# Directional distance curves
agg_rt3 = agg_rt2.copy()
agg_rt3["direction"] = np.where(
    agg_rt3["to_release"].astype(int) >= agg_rt3["from_release"].astype(int),
    "future_or_same",
    "past",
)

curve_dir = (
    agg_rt3[agg_rt3["abs_delta"] > 0]
    .groupby(["direction", "abs_delta"], as_index=False)
    .agg(
        mean=("frac_roundtrip_given_success", "mean"),
        std=("frac_roundtrip_given_success", "std"),
        n=("abs_delta", "count"),
    )
)
curve_dir["sem"] = curve_dir["std"] / np.sqrt(curve_dir["n"].replace(0, np.nan))

fig3, ax3 = plt.subplots(1, 1, figsize=(7.8, 4.2), constrained_layout=True)
for direction, color in [("future_or_same", MANUSCRIPT_COLORS["1→1"]), ("past", MANUSCRIPT_COLORS["1→0"])]:
    sub = curve_dir[curve_dir["direction"] == direction]
    if sub.empty:
        continue
    ax3.errorbar(
        sub["abs_delta"],
        sub["mean"],
        yerr=sub["sem"],
        fmt="-o",
        ms=3,
        lw=1.4,
        color=color,
        label=direction.replace("_", " "),
    )
ax3.set_ylim(0, 1)
ax3.set_xlabel("|Δ release|")
ax3.set_ylabel("Round-trip recovery | forward success")
ax3.set_title("Directional round-trip decay (future vs past)")
ax3.legend(frameon=True, fontsize=9)

written3 = save_figure(fig3, "fig_time_travel_roundtrip_directional_decay.pdf", ctx, formats=("pdf",))
print("Saved:", written3["pdf"])

# Worst-case pairs table
pairs = agg_rt3.copy()
pairs["irreversibility"] = 1.0 - pairs["frac_roundtrip_given_success"]
worst = pairs.sort_values(["irreversibility", "abs_delta"], ascending=[False, False]).head(25)
out_worst = MANUSCRIPT_TABLES / "time_travel_roundtrip_worst_pairs.csv"
atomic_write_dataframe_csv(worst, out_worst, index=False)
atomic_write_dataframe_csv(worst, (ctx.experiment_outputs / "tables" / out_worst.name), index=False)
print("Wrote:", out_worst)
display(worst.head(10))
